# Random Forest Classification: Dry Bean Dataset

## Objectives
1. Compare a baseline Decision Tree with a Random Forest.
2. Examine the effect of `max_depth`.
3. Examine the effect of `n_estimators`.
4. Evaluate Accuracy, macro Precision, and macro Recall.

The workbook contains 13,611 records, 16 numerical predictors, and the categorical target `Class`. The experiment uses an 80/20 stratified train-test split and a fixed random seed.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay, classification_report

df = pd.read_excel("Dry_Bean_Dataset.xlsx")
print("Shape:", df.shape)
display(df.head())
print("Missing values:", df.isna().sum().sum())
print(df["Class"].value_counts())

Shape: (13611, 17)
Missing values: 0


In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training:", X_train.shape, "Testing:", X_test.shape)

Training: (10888, 16) Testing: (2723, 16)


In [ ]:
def evaluate(model):
    pred = model.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (macro)": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, pred, average="macro", zero_division=0)
    }

tree = DecisionTreeClassifier(random_state=42)
forest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
tree.fit(X_train, y_train)
forest.fit(X_train, y_train)
baseline = pd.DataFrame({"Decision Tree": evaluate(tree), "Random Forest": evaluate(forest)})
display(baseline.round(4))
baseline.plot(kind="bar", ylim=(0, 1.05), figsize=(8, 4), title="Baseline model comparison")
plt.ylabel("Score"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

                 Decision Tree  Random Forest
Accuracy                0.8917          0.9218
Precision (macro)       0.9084          0.9362
Recall (macro)          0.9088          0.9323


In [ ]:
depth_values = [2, 4, 6, 8, 10, 15, None]
depth_rows = []
for depth in depth_values:
    model = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    depth_rows.append({"max_depth": str(depth), **evaluate(model)})
depth_results = pd.DataFrame(depth_rows)
display(depth_results.round(4))
depth_results.set_index("max_depth").plot(marker="o", figsize=(8, 4), ylim=(0.75, 1.01), title="Effect of max_depth")
plt.ylabel("Score"); plt.grid(alpha=0.25); plt.tight_layout(); plt.show()

  max_depth  Accuracy  Precision (macro)  Recall (macro)
0         2    0.7675              0.5637          0.6325
1         4    0.8821              0.9061          0.8822
2         6    0.8920              0.9070          0.8964
3         8    0.9141              0.9311          0.9239
4        10    0.9207              0.9356          0.9306
5        15    0.9188              0.9337          0.9293
6      None    0.9218              0.9362          0.9323


In [ ]:
tree_counts = [10, 25, 50, 100, 200, 300]
tree_rows = []
for count in tree_counts:
    model = RandomForestClassifier(n_estimators=count, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    tree_rows.append({"n_estimators": count, **evaluate(model)})
estimator_results = pd.DataFrame(tree_rows)
display(estimator_results.round(4))
estimator_results.set_index("n_estimators").plot(marker="o", figsize=(8, 4), ylim=(0.75, 1.01), title="Effect of n_estimators")
plt.ylabel("Score"); plt.grid(alpha=0.25); plt.tight_layout(); plt.show()

   n_estimators  Accuracy  Precision (macro)  Recall (macro)
0            10    0.9130              0.9290          0.9255
1            25    0.9181              0.9322          0.9284
2            50    0.9210              0.9351          0.9313
3           100    0.9218              0.9362          0.9323
4           200    0.9221              0.9359          0.9329
5           300    0.9218              0.9361          0.9325


In [ ]:
ConfusionMatrixDisplay.from_estimator(forest, X_test, y_test, xticks_rotation=40)
plt.title("Random Forest confusion matrix"); plt.tight_layout(); plt.show()
print(classification_report(y_test, forest.predict(X_test), zero_division=0))

              precision    recall  f1-score   support
BARBUNYA          0.87      0.90      0.88       264
BOMBAY            0.99      1.00      0.99       104
CALI              0.96      0.96      0.96       326
DERMASON          0.91      0.93      0.92       709
HOROZ             0.97      0.98      0.97       386
SEKER             0.96      0.92      0.94       405
SIRA              0.96      0.92      0.94       529

accuracy                               0.92      2723
macro avg         0.95      0.94      0.94      2723
weighted avg      0.92      0.92      0.92      2723
